# Natural-Language Satellite Image Analyst: Water Change Walkthrough

This notebook provides a defensible, step-by-step walkthrough of the Phase 2 geospatial pipeline for surface water change detection.

**Dataset**: Operational Sentinel-2 L2A BOA Reflectance over **Raipur, Chhattisgarh** ($30 \times 30\text{ km}$, $900\text{ km}^2$, Post-Monsoon Oct–Nov 2023 vs 2025).
**Sensor Calibration**: SCL cloud/shadow filtering and ESA baseline $\ge 04.00$ `BOA_ADD_OFFSET = -1000` radiometric correction.
**Equal-Area Projection**: UTM Zone 44N (EPSG:32644) at 10m native pixel resolution.

## 1. Load Pre-Computed Sentinel-2 L2A Composites
We read the same-season median composites generated from 33 Sentinel-2 scenes in 2023 and 22 scenes in 2025.

In [1]:
from pathlib import Path

import numpy as np
import rasterio

# Paths to cached composites (supports execution from root or notebooks/)
base_dir = Path("outputs") if Path("outputs").exists() else Path("../outputs")
g23_p = base_dir / "raipur_2023_post_monsoon_green.tif"
s23_p = base_dir / "raipur_2023_post_monsoon_swir1.tif"
g25_p = base_dir / "raipur_2025_post_monsoon_green.tif"
s25_p = base_dir / "raipur_2025_post_monsoon_swir1.tif"

with rasterio.open(g23_p) as src:
    green_2023 = src.read(1)
    crs = src.crs
    transform = src.transform
    shape = green_2023.shape

with rasterio.open(s23_p) as src:
    swir1_2023 = src.read(1)
with rasterio.open(g25_p) as src:
    green_2025 = src.read(1)
with rasterio.open(s25_p) as src:
    swir1_2025 = src.read(1)

print(f"Raster Grid Dimensions: {shape} ({shape[0] * shape[1]:,} pixels)")
print(f"Coordinate Reference System: {crs}")
print(f"Spatial Resolution: {abs(transform.a):.1f}m x {abs(transform.e):.1f}m per pixel")
print(
    f"2023 Green BOA Reflectance [p2, p98]: [{np.nanpercentile(green_2023, 2):.4f}, {np.nanpercentile(green_2023, 98):.4f}]"
)
print(
    f"2025 Green BOA Reflectance [p2, p98]: [{np.nanpercentile(green_2025, 2):.4f}, {np.nanpercentile(green_2025, 98):.4f}]"
)

Raster Grid Dimensions: (3000, 3025) (9,075,000 pixels)
Coordinate Reference System: EPSG:32644
Spatial Resolution: 10.0m x 10.0m per pixel
2023 Green BOA Reflectance [p2, p98]: [0.0507, 0.1453]
2025 Green BOA Reflectance [p2, p98]: [0.0472, 0.1396]


## 2. Compute Modified Normalized Difference Water Index (MNDWI)
$$\text{MNDWI} = \frac{\text{Green} - \text{SWIR1}}{\text{Green} + \text{SWIR1}} = \frac{\text{B03} - \text{B11}}{\text{B03} + \text{B11}}$$
MNDWI suppresses built-up urban features and soil noise much more effectively than McFeeters NDWI.

In [2]:
from satellite_analyst.analysis.indices import compute_mndwi

mndwi_2023 = compute_mndwi(green_2023, swir1_2023)
mndwi_2025 = compute_mndwi(green_2025, swir1_2025)

print(
    f"2023 MNDWI Range: [{np.nanmin(mndwi_2023):.3f}, {np.nanmax(mndwi_2023):.3f}] (Mean: {np.nanmean(mndwi_2023):.3f})"
)
print(
    f"2025 MNDWI Range: [{np.nanmin(mndwi_2025):.3f}, {np.nanmax(mndwi_2025):.3f}] (Mean: {np.nanmean(mndwi_2025):.3f})"
)

2023 MNDWI Range: [-0.702, 0.799] (Mean: -0.366)
2025 MNDWI Range: [-0.852, 0.811] (Mean: -0.371)


## 3. Histogram Bimodality Verification & Shared Threshold Policy
When water constitutes a small fraction of the total landscape ($< 5\%$), standard Otsu thresholding erroneously bisects the unimodal dry land / agricultural peak.
Our pipeline computes Otsu between-class variance ratio $\eta$ and mean separation $\Delta\mu$. If the distribution is not bimodal, it defensively falls back to `0.0`.

In [3]:
from satellite_analyst.analysis.water import compute_otsu_threshold, segment_dual_period_water

valid_pooled = np.concatenate(
    [mndwi_2023[np.isfinite(mndwi_2023)], mndwi_2025[np.isfinite(mndwi_2025)]]
)
raw_otsu, is_bimodal, eta = compute_otsu_threshold(valid_pooled, bimodality_min_ratio=0.70)

print(f"Pooled Otsu Split: {raw_otsu:.4f}")
print(f"Variance Separation Ratio (eta): {eta:.4f} (Required: >= 0.7000)")
print(f"Is Bimodal Land-Water Distribution: {is_bimodal}")
if not is_bimodal:
    print("  --> Defense Triggered: Fallback to fixed threshold 0.0000 to prevent false change.")

mask_2023, mask_2025 = segment_dual_period_water(
    mndwi_2023, mndwi_2025, policy="shared", min_area_ha=0.1
)

area_23_ha = np.sum(mask_2023.mask) * 0.01
area_25_ha = np.sum(mask_2025.mask) * 0.01
print(f"\nApplied Strategy: {mask_2023.method} (Threshold: {mask_2023.threshold:.4f})")
print(
    f"2023 Segmented Surface Water: {area_23_ha:,.2f} ha ({mask_2023.water_fraction * 100:.2f}% of AOI)"
)
print(
    f"2025 Segmented Surface Water: {area_25_ha:,.2f} ha ({mask_2025.water_fraction * 100:.2f}% of AOI)"
)

Pooled Otsu Split: -0.2852
Variance Separation Ratio (eta): 0.4979 (Required: >= 0.7000)
Is Bimodal Land-Water Distribution: False
  --> Defense Triggered: Fallback to fixed threshold 0.0000 to prevent false change.

Applied Strategy: shared_otsu_fallback (Threshold: 0.0000)
2023 Segmented Surface Water: 1,674.99 ha (1.85% of AOI)
2025 Segmented Surface Water: 1,721.65 ha (1.90% of AOI)


## 4. 3-Class Change Classification
We categorize transitions into:
- **Persistent (3)**: Water present in both 2023 and 2025
- **Gained (1)**: Dry in 2023, water in 2025
- **Lost (2)**: Water in 2023, dry in 2025
- **Dry (0)**: Non-water in both periods
- **NoData (255)**: Invalid / low-confidence observations in either period

In [4]:
from satellite_analyst.analysis.change import WaterChangeClass, detect_water_change

change_raster = detect_water_change(mask_2023, mask_2025, min_valid_obs=2)

count_dry = int(np.sum(change_raster == WaterChangeClass.DRY))
count_gained = int(np.sum(change_raster == WaterChangeClass.GAINED))
count_lost = int(np.sum(change_raster == WaterChangeClass.LOST))
count_persistent = int(np.sum(change_raster == WaterChangeClass.PERSISTENT))

print("Pixel Transitions (1 px = 0.01 ha):")
print(f"  Persistent Water: {count_persistent:,} px ({count_persistent * 0.01:,.2f} ha)")
print(f"  Gained Water:     {count_gained:,} px ({count_gained * 0.01:,.2f} ha)")
print(f"  Lost Water:       {count_lost:,} px ({count_lost * 0.01:,.2f} ha)")
print(f"  Dry Land:         {count_dry:,} px ({count_dry * 0.01:,.2f} ha)")

Pixel Transitions (1 px = 0.01 ha):
  Persistent Water: 144,536 px (1,445.36 ha)
  Gained Water:     27,629 px (276.29 ha)
  Lost Water:       22,963 px (229.63 ha)
  Dry Land:         8,879,872 px (88,798.72 ha)


## 5. Connected-Component Waterbody Tracking & Statistics
We group water bodies by connected components of $\text{union}(\text{Water}_{2023}, \text{Water}_{2025})$, assign deterministic geographic hash IDs, and compute regional totals.

In [5]:
import pandas as pd

from satellite_analyst.analysis.stats import compute_summary_stats
from satellite_analyst.analysis.vectorize import extract_waterbody_records

wb_records, wb_gdf = extract_waterbody_records(
    mask_2023.mask, mask_2025.mask, transform=transform, crs="EPSG:32644", min_area_ha=0.1
)

stats = compute_summary_stats(change_raster, wb_records, pixel_size_m=10.0)
print(f"Total Tracked Waterbody Features: {len(wb_records):,}")
print(f"Net Regional Water Change: {stats.net_change_ha:+.2f} ha ({stats.percent_change:+.2f}%)")

df_wb = pd.DataFrame([r.to_dict() for r in wb_records])
print("\nTop 10 Waterbodies with Largest Dynamics:")
print(
    df_wb[
        [
            "waterbody_id",
            "centroid_lon",
            "centroid_lat",
            "area_period_a_ha",
            "area_period_b_ha",
            "net_change_ha",
            "status",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

# Sanity check on Budha Talab (Vivekananda Sarovar in central Raipur)
budha = df_wb[df_wb["waterbody_id"] == "wb_dca58935"]
print("\nSanity Check: Budha Talab / Vivekananda Sarovar:")
print(
    budha[
        [
            "waterbody_id",
            "centroid_lon",
            "centroid_lat",
            "area_period_a_ha",
            "area_period_b_ha",
            "status",
        ]
    ].to_string(index=False)
)

Total Tracked Waterbody Features: 1,403
Net Regional Water Change: +46.66 ha (+2.79%)

Top 10 Waterbodies with Largest Dynamics:
waterbody_id  centroid_lon  centroid_lat  area_period_a_ha  area_period_b_ha  net_change_ha      status
 wb_46817168      81.60958      21.15567             53.35             40.48         -12.87      shrunk
 wb_32608d46      81.60562      21.19380             37.58             47.24           9.66    expanded
 wb_f980a735      81.55202      21.34782              9.87             18.44           8.57    expanded
 wb_97aea60b      81.74644      21.24061             49.79             41.28          -8.51      shrunk
 wb_bc6aed59      81.60028      21.36007             17.52              9.56          -7.96      shrunk
 wb_d5098d73      81.50356      21.29706             34.93             42.85           7.92    expanded
 wb_d0798956      81.70294      21.35121              0.52              7.53           7.01    expanded
 wb_ff4cc75a      81.65011      21.2001